# Track 1 EDA — National Vaccination Trends + CDC Communication Signal

**Research question:** How did CDC communication sentiment relate to COVID-19 vaccination uptake during the post-rollout period (Jan 2021 – Oct 2022)?

**This notebook covers:**
1. Data loading and cleaning
2. Vaccination uptake curves by age group
3. COVID-19 case burden by age group over time
4. CDC communication sentiment timeline
5. Overlay: CDC signal vs. national vaccination rate
6. Correlation analysis

**Note:** This is national-level data. County-level Virginia analysis is in `02_EDA_virginia_counties.ipynb` (requires COVIDVaccinationsUSCounty.csv).

## Cell 1 — Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Imports OK')

Imports OK


## Cell 2 — Load and clean vaccination data

In [5]:
# ── Load ──────────────────────────────────────────────────────────────────────
VAX_PATH = '/workspace/COVIDResearch/COVIDVaccinationsUSCounty.csv'

vax = pd.read_csv(VAX_PATH)

# ── Parse dates ───────────────────────────────────────────────────────────────
vax['date'] = pd.to_datetime(vax['Date'], format='mixed')

# ── Strip % signs and convert to float ────────────────────────────────────────
# These came in as strings like '71.1%' — need to be numeric for plotting
vax['dose1_pct'] = (
    vax['Administered_Dose1_pct_agegroup']
    .str.replace('%', '', regex=False)
    .astype(float)
)
vax['complete_pct'] = (
    vax['Series_Complete_Pop_pct_agegroup']
    .str.replace('%', '', regex=False)
    .astype(float)
)

# ── Add month period for aggregation ──────────────────────────────────────────
vax['month'] = vax['date'].dt.to_period('M')

# ── Filter to Jan 2021 onward (vaccine era) ───────────────────────────────────
vax = vax[vax['date'] >= '2021-01-01'].copy()

# ── Exclude under-2 and 2-4 — these had very late/limited eligibility ─────────
# Keep them if you want but they distort the main story
EXCLUDE_GROUPS = ['<2 Years', '2 - 4 Years']
vax_main = vax[~vax['AgeGroupVacc'].isin(EXCLUDE_GROUPS)].copy()

print('Shape (all):', vax.shape)
print('Shape (main age groups):', vax_main.shape)
print('Date range:', vax['date'].min().date(), 'to', vax['date'].max().date())
print('\nAge groups kept:', sorted(vax_main['AgeGroupVacc'].unique()))
print('\nNull check:')
print(vax_main[['date','dose1_pct','complete_pct','7-day_avg_group_cases_per_100k']].isnull().sum())
vax_main.head()

KeyError: 'Administered_Dose1_pct_agegroup'

## Cell 3 — Load CDC communications data

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
CDC_PATH     = 'data/raw/cdc_key_communications.csv'
CDC_MON_PATH = 'data/processed/cdc_monthly_sentiment.csv'

cdc = pd.read_csv(CDC_PATH)
cdc['date'] = pd.to_datetime(cdc['date'])
cdc['month'] = pd.PeriodIndex(cdc['month'], freq='M')

cdc_m = pd.read_csv(CDC_MON_PATH)
cdc_m['month'] = pd.PeriodIndex(cdc_m['month'], freq='M')
# Convert month to datetime for plotting
cdc_m['month_dt'] = cdc_m['month'].dt.to_timestamp()

print('CDC events:', len(cdc))
print('Date range:', cdc['date'].min().date(), 'to', cdc['date'].max().date())
print('\nEvent types:')
print(cdc['event_type'].value_counts())
print('\nValence counts:')
print(cdc['valence'].value_counts().sort_index())
print('\nMonthly aggregated:')
cdc_m

## Cell 4 — Fig 1: Vaccination uptake curves by age group

In [ ]:
# Color palette — one color per age group
AGE_COLORS = {
    '5 - 11 Years':  '#a8d5e2',
    '12 - 17 Years': '#6cb4d9',
    '18 - 24 Years': '#f4a261',
    '25 - 49 Years': '#e76f51',
    '50 - 64 Years': '#2a9d8f',
    '65+ Years':     '#264653',
}

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
fig.suptitle('U.S. COVID-19 Vaccination Uptake by Age Group\nJan 2021 – Oct 2022',
             fontsize=14, fontweight='bold', y=0.98)

# ── Key CDC event dates to annotate ───────────────────────────────────────────
key_events = [
    ('2021-04-13', 'J&J\npause',    'red'),
    ('2021-07-27', 'Mask\nreversal', 'orange'),
    ('2021-08-23', 'FDA full\napproval', 'green'),
    ('2021-11-29', 'Omicron\ndetected', 'red'),
    ('2022-09-01', 'Bivalent\nbooster', 'green'),
]

for ax_idx, (metric, ylabel, title_suffix) in enumerate([
    ('dose1_pct',    '% Received at Least 1 Dose', 'At Least One Dose'),
    ('complete_pct', '% Fully Vaccinated',          'Series Complete'),
]):
    ax = axes[ax_idx]

    for age_group, color in AGE_COLORS.items():
        df_grp = vax_main[vax_main['AgeGroupVacc'] == age_group].sort_values('date')
        ax.plot(df_grp['date'], df_grp[metric],
                label=age_group, color=color, linewidth=2)

    # Annotate key CDC events
    for ev_date, ev_label, ev_color in key_events:
        ax.axvline(pd.Timestamp(ev_date), color=ev_color,
                   linestyle='--', linewidth=0.9, alpha=0.7)
        if ax_idx == 0:  # only label on top panel to avoid clutter
            ax.text(pd.Timestamp(ev_date), ax.get_ylim()[1] * 0.97,
                    ev_label, fontsize=7, color=ev_color,
                    ha='center', va='top', rotation=0)

    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title_suffix, fontsize=11, pad=4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3)

# Single legend outside panels
handles = [mpatches.Patch(color=c, label=g) for g, c in AGE_COLORS.items()]
fig.legend(handles=handles, title='Age Group', loc='center right',
           bbox_to_anchor=(1.13, 0.5), fontsize=9, title_fontsize=9)

axes[1].set_xlabel('Date', fontsize=11)
plt.tight_layout()
plt.savefig('figures/fig1_vax_by_age_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/fig1_vax_by_age_group.png')

## Cell 5 — Fig 2: COVID-19 cases per 100k by age group (wave identification)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for age_group, color in AGE_COLORS.items():
    df_grp = vax_main[vax_main['AgeGroupVacc'] == age_group].sort_values('date')
    ax.plot(df_grp['date'], df_grp['7-day_avg_group_cases_per_100k'],
            label=age_group, color=color, linewidth=1.8)

# Shade the major waves
waves = [
    ('2021-06-20', '2021-09-30', 'Delta wave',   '#f4a261', 0.08),
    ('2021-11-15', '2022-02-28', 'Omicron wave', '#e76f51', 0.08),
    ('2022-06-01', '2022-08-31', 'BA.4/BA.5',    '#6cb4d9', 0.08),
]
for start, end, label, color, alpha in waves:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               color=color, alpha=alpha, label=label)
    ax.text(pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start))/2,
            ax.get_ylim()[1] * 0.92 if ax.get_ylim()[1] > 0 else 250,
            label, ha='center', fontsize=8, color='gray')

ax.set_title('7-Day Average COVID-19 Cases per 100k by Age Group\nJan 2021 – Oct 2022',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Cases per 100k (7-day avg)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.grid(axis='y', alpha=0.3)

handles = [mpatches.Patch(color=c, label=g) for g, c in AGE_COLORS.items()]
ax.legend(handles=handles, title='Age Group', loc='upper left',
          fontsize=8, title_fontsize=8)

plt.tight_layout()
plt.savefig('figures/fig2_cases_by_age_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/fig2_cases_by_age_group.png')

## Cell 6 — Fig 3: CDC communication sentiment timeline

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle('CDC COVID-19 Communications — Sentiment Over Time\nJan 2021 – Oct 2022',
             fontsize=13, fontweight='bold')

# ── Top panel: individual events as scatter ────────────────────────────────────
ax1 = axes[0]
type_colors = {
    'approval':     '#2a9d8f',
    'guidance':     '#264653',
    'data_release': '#6cb4d9',
    'warning':      '#e63946',
    'funding':      '#f4a261',
    'statement':    '#adb5bd',
}
for ev_type, color in type_colors.items():
    subset = cdc[cdc['event_type'] == ev_type]
    ax1.scatter(subset['date'], subset['valence'],
                color=color, label=ev_type, s=60, zorder=3, alpha=0.85)

ax1.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax1.set_ylabel('Valence (+1 / 0 / -1)', fontsize=10)
ax1.set_yticks([-1, 0, 1])
ax1.set_yticklabels(['Negative\n(-1)', 'Neutral\n(0)', 'Positive\n(+1)'], fontsize=8)
ax1.set_title('Individual Communications', fontsize=10, pad=3)
ax1.legend(title='Type', loc='lower right', fontsize=7,
           title_fontsize=7, ncol=3)
ax1.grid(axis='y', alpha=0.25)

# ── Bottom panel: monthly avg valence bar chart ────────────────────────────────
ax2 = axes[1]
bar_colors = ['#e63946' if v < 0 else '#2a9d8f' if v > 0 else '#adb5bd'
              for v in cdc_m['avg_valence']]
bars = ax2.bar(cdc_m['month_dt'], cdc_m['avg_valence'],
               color=bar_colors, width=20, alpha=0.85, zorder=3)
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')

# Label n_communications above each bar
for _, row in cdc_m.iterrows():
    ax2.text(row['month_dt'], row['avg_valence'] + (0.05 if row['avg_valence'] >= 0 else -0.12),
             f"n={int(row['n_communications'])}",
             ha='center', fontsize=6.5, color='#555')

ax2.set_ylabel('Monthly Avg Valence', fontsize=10)
ax2.set_xlabel('Month', fontsize=10)
ax2.set_title('Monthly Average (bars = avg valence, red = net negative, green = net positive)',
              fontsize=9, pad=3)
ax2.set_ylim(-1.4, 1.4)
ax2.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig('figures/fig3_cdc_sentiment_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/fig3_cdc_sentiment_timeline.png')

## Cell 7 — Build merged monthly dataset

In [ ]:
# ── Aggregate vaccination data to monthly, keep all ages + an overall avg ──────
# 'Overall' = mean across all main age groups each month
vax_monthly_all = (
    vax_main
    .groupby('month')
    .agg(
        avg_complete_pct  = ('complete_pct', 'mean'),
        avg_dose1_pct     = ('dose1_pct', 'mean'),
        avg_cases_per100k = ('7-day_avg_group_cases_per_100k', 'mean'),
    )
    .reset_index()
)

# ── Also keep 65+ separately — most relevant for VA county analysis later ──────
vax_65plus = (
    vax_main[vax_main['AgeGroupVacc'] == '65+ Years']
    .groupby('month')
    .agg(complete_pct_65plus=('complete_pct', 'mean'))
    .reset_index()
)

# ── Merge vaccination monthly + CDC monthly ────────────────────────────────────
merged = vax_monthly_all.merge(cdc_m[['month','avg_valence','n_communications',
                                       'positive_comms','negative_comms']],
                                on='month', how='left')
merged = merged.merge(vax_65plus, on='month', how='left')
merged['month_dt'] = merged['month'].dt.to_timestamp()

# ── Month-over-month change in vaccination rate ────────────────────────────────
merged['complete_pct_change'] = merged['avg_complete_pct'].diff()

# ── Lagged CDC valence (T-1 and T-2) ──────────────────────────────────────────
merged['valence_lag1'] = merged['avg_valence'].shift(1)
merged['valence_lag2'] = merged['avg_valence'].shift(2)

print('Merged dataset shape:', merged.shape)
print('Columns:', merged.columns.tolist())
print()
merged.head(10)

## Cell 8 — Fig 4: Overlay — CDC sentiment vs. national vaccination rate

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))

# ── Left axis: vaccination rate ────────────────────────────────────────────────
color_vax = '#264653'
ax1.plot(merged['month_dt'], merged['avg_complete_pct'],
         color=color_vax, linewidth=2.5, label='Avg % Fully Vaccinated (all ages)')
ax1.set_ylabel('% Fully Vaccinated (avg across age groups)', color=color_vax, fontsize=10)
ax1.tick_params(axis='y', labelcolor=color_vax)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax1.set_ylim(0, 85)

# ── Right axis: CDC monthly valence ───────────────────────────────────────────
ax2 = ax1.twinx()
color_cdc = '#e63946'

# Fill under/over 0
ax2.fill_between(merged['month_dt'], merged['avg_valence'], 0,
                 where=merged['avg_valence'] >= 0,
                 color='#2a9d8f', alpha=0.25, label='Positive CDC month')
ax2.fill_between(merged['month_dt'], merged['avg_valence'], 0,
                 where=merged['avg_valence'] < 0,
                 color='#e63946', alpha=0.25, label='Negative CDC month')
ax2.plot(merged['month_dt'], merged['avg_valence'],
         color=color_cdc, linewidth=1.5, linestyle='--', marker='o', markersize=4)
ax2.set_ylabel('CDC Monthly Avg Valence', color=color_cdc, fontsize=10)
ax2.tick_params(axis='y', labelcolor=color_cdc)
ax2.set_ylim(-2, 2)
ax2.axhline(0, color='gray', linewidth=0.6, linestyle=':')

# ── Annotate key negative CDC events ──────────────────────────────────────────
neg_events = cdc[cdc['valence'] == -1][['date', 'title']].copy()
notable = [
    ('2021-04-13', 'J&J\npause'),
    ('2021-07-27', 'Mask\nreversal'),
    ('2021-11-29', 'Omicron\ndetected'),
    ('2022-01-07', 'Omicron\n95% of cases'),
]
for ev_date, ev_label in notable:
    ax1.axvline(pd.Timestamp(ev_date), color='#e63946',
                linewidth=0.8, linestyle=':', alpha=0.6)
    ax1.text(pd.Timestamp(ev_date), 5, ev_label,
             fontsize=7, color='#e63946', ha='center', va='bottom')

ax1.set_title('CDC Communication Sentiment vs. National Vaccination Rate\nJan 2021 – Oct 2022',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Month', fontsize=10)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

ax1.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('figures/fig4_cdc_vs_vaccination_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/fig4_cdc_vs_vaccination_overlay.png')

## Cell 9 — Fig 5: Month-over-month vaccination rate change vs. CDC valence

In [ ]:
# Drop first row (NaN from .diff()) and months with no CDC data
scatter_df = merged.dropna(subset=['complete_pct_change', 'avg_valence']).copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('CDC Valence vs. Monthly Vaccination Rate Change\n(Same month, 1-month lag, 2-month lag)',
             fontsize=12, fontweight='bold')

for i, (lag_col, lag_label) in enumerate([
    ('avg_valence',  'Same month (T)'),
    ('valence_lag1', '1-month lag (T-1)'),
    ('valence_lag2', '2-month lag (T-2)'),
]):
    ax = axes[i]
    plot_df = scatter_df.dropna(subset=[lag_col, 'complete_pct_change'])

    # Color points by valence direction
    point_colors = ['#e63946' if v < 0 else '#2a9d8f' if v > 0 else '#adb5bd'
                    for v in plot_df[lag_col]]

    ax.scatter(plot_df[lag_col], plot_df['complete_pct_change'],
               c=point_colors, s=55, alpha=0.85, zorder=3)

    # Add regression line
    if len(plot_df) > 2:
        z = np.polyfit(plot_df[lag_col], plot_df['complete_pct_change'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(plot_df[lag_col].min(), plot_df[lag_col].max(), 100)
        ax.plot(x_line, p(x_line), color='#264653', linewidth=1.5, linestyle='--')

        corr = plot_df[lag_col].corr(plot_df['complete_pct_change'])
        ax.text(0.05, 0.93, f'r = {corr:.3f}', transform=ax.transAxes,
                fontsize=10, fontweight='bold',
                color='#264653' if abs(corr) > 0.3 else 'gray')

    ax.axhline(0, color='gray', linewidth=0.6, linestyle=':')
    ax.axvline(0, color='gray', linewidth=0.6, linestyle=':')
    ax.set_xlabel(f'CDC Valence ({lag_label})', fontsize=9)
    ax.set_ylabel('Monthly Δ% Fully Vaccinated' if i == 0 else '', fontsize=9)
    ax.set_title(lag_label, fontsize=10)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('figures/fig5_lag_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/fig5_lag_scatter.png')
print()
print('Interpretation guide:')
print('  r > +0.3 = positive CDC months correlate with faster vaccination uptake')
print('  r < -0.3 = negative CDC months correlate with faster uptake (possibly fear-driven)')
print('  Strongest r across the 3 lags tells you the most likely response window')

## Cell 10 — Summary statistics and key findings

In [ ]:
print('=' * 60)
print('TRACK 1 EDA SUMMARY')
print('=' * 60)

print('\n── Vaccination uptake (avg across main age groups) ───────')
print(f"  Jan 2021 start:  {merged['avg_complete_pct'].iloc[0]:.1f}%")
print(f"  Peak:            {merged['avg_complete_pct'].max():.1f}% "
      f"({merged.loc[merged['avg_complete_pct'].idxmax(), 'month']})")
print(f"  Oct 2022 end:    {merged['avg_complete_pct'].iloc[-1]:.1f}%")

print('\n── 65+ vs younger age groups ─────────────────────────────')
grp65 = vax_main[vax_main['AgeGroupVacc'] == '65+ Years']
grp18 = vax_main[vax_main['AgeGroupVacc'] == '18 - 24 Years']
for label, grp in [('65+ Years', grp65), ('18-24 Years', grp18)]:
    peak = grp['complete_pct'].max()
    peak_date = grp.loc[grp['complete_pct'].idxmax(), 'date'].strftime('%b %Y')
    print(f"  {label}: peak {peak:.1f}% ({peak_date})")

print('\n── CDC communications ────────────────────────────────────')
print(f"  Total events logged:    {len(cdc)}")
print(f"  Positive (valence=+1):  {(cdc['valence']==1).sum()}")
print(f"  Neutral  (valence= 0):  {(cdc['valence']==0).sum()}")
print(f"  Negative (valence=-1):  {(cdc['valence']==-1).sum()}")
print(f"  Most negative month:    "
      f"{cdc_m.loc[cdc_m['avg_valence'].idxmin(), 'month']} "
      f"(avg valence {cdc_m['avg_valence'].min():.2f})")

print('\n── Lag correlations (CDC valence → Δ% vaccinated) ────────')
for lag_col, lag_label in [
    ('avg_valence',  'Same month'),
    ('valence_lag1', '1-month lag'),
    ('valence_lag2', '2-month lag'),
]:
    df_tmp = merged.dropna(subset=[lag_col, 'complete_pct_change'])
    r = df_tmp[lag_col].corr(df_tmp['complete_pct_change'])
    print(f"  {lag_label}: r = {r:.3f}")

print()
print('Next step: Run 02_EDA_virginia_counties.ipynb once')
print('COVIDVaccinationsUSCounty.csv is in data/raw/')
print('=' * 60)

## Cell 11 — Save merged monthly dataset for regression notebook

In [ ]:
# Drop the Period dtype column before saving (not CSV-serializable)
save_cols = [c for c in merged.columns if c != 'month']
merged[save_cols].to_csv('data/processed/national_monthly_panel.csv', index=False)
print('Saved: data/processed/national_monthly_panel.csv')
print('Columns:', save_cols)
print('Shape:', merged.shape)